In [ ]:
!mkdir data
%cd data
!curl -L -o ./landscape-pictures.zip  https://www.kaggle.com/api/v1/datasets/download/arnaud58/landscape-pictures
!unzip ./landscape-pictures.zip
!rm ./landscape-pictures.zip

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, utils
from PIL import Image
import os
import glob
import matplotlib.pyplot as plt
import numpy as np

from torchsummary import summary

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Custom Dataset Class (Loads images directly from 'landscape-pictures')
class CustomImageDataset(Dataset):
    def __init__(self, image_dir, transform=None):
        self.image_dir = image_dir
        self.transform = transform
        self.image_paths = sorted(glob.glob(os.path.join(image_dir, '*.jpg')))

        if len(self.image_paths) == 0:
            raise ValueError(f"No images found in directory: {image_dir}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image

# Dataset Path
image_dir = './data'  

# Image Transformations
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    # transforms.CenterCrop(64),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Load Dataset
dataset = CustomImageDataset(image_dir=image_dir, transform=transform)
print(f"Total images found: {len(dataset)}")

# DataLoader
batch_size = 64  
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [ ]:
class Encoder(nn.Module):
    def __init__(self, nc: int = 3, ngf: int = 64, nz: int = 4):
        super(Encoder, self).__init__()
        self.main = nn.Sequential(
            # inverse of last TConv: (ngf -> nc, 3x3, s=1, p=1)
            nn.Conv2d(nc, ngf, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.LeakyReLU(0.2, inplace=True),

            # inverse of TConv (ngf*2 <- ngf, up x2)
            nn.Conv2d(ngf, ngf * 2, kernel_size=4, stride=2, padding=1, bias=False),  # 256 -> 128
            nn.BatchNorm2d(ngf * 2),
            nn.LeakyReLU(0.2, inplace=True),

            # inverse of TConv (ngf*4 <- ngf*2, up x2)
            nn.Conv2d(ngf * 2, ngf * 4, kernel_size=4, stride=2, padding=1, bias=False),  # 128 -> 64
            nn.BatchNorm2d(ngf * 4),
            nn.LeakyReLU(0.2, inplace=True),

            # inverse of TConv (ngf*8 <- ngf*4, up x2)
            nn.Conv2d(ngf * 4, ngf * 8, kernel_size=4, stride=2, padding=1, bias=False),  # 64 -> 32
            nn.BatchNorm2d(ngf * 8),
            nn.LeakyReLU(0.2, inplace=True),

            # inverse of first TConv: (nz <- ngf*8, 3x3, s=1, p=1) keeps 32x32
            nn.Conv2d(ngf * 8, nz*2, kernel_size=3, stride=1, padding=1, bias=False),
            # no BN/act—treat as latent
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.main(x)  # (B, 4, 32, 32)
    
# Define Generator
class Decoder(nn.Module):
    def __init__(self, nz=4, ngf=64, nc=3):
        super(Decoder, self).__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(nz, ngf * 8, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),

            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),

            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),

            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),

            nn.ConvTranspose2d(ngf, nc, 3, 1, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, input):
        return self.main(input)

# Define Discriminator
class Discriminator(nn.Module):
    def __init__(self, nc=3, ndf=64):
        super(Discriminator, self).__init__()
        self.main = nn.Sequential(
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(ndf * 8, ndf * 16, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 16),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf * 16, 1, 8, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, input):
        return self.main(input).view(-1, 1)  

In [ ]:
# Initialize models
netG = Decoder().to(device)
netD = Encoder().to(device)

summary(netG, (4, 32, 32))
summary(netD, (3, 256, 256))

In [ ]:
# Loss and Optimizer
criterion = nn.BCELoss()
lr = 0.0002
beta1 = 0.5
optimizerD = optim.Adam(netD.parameters(), lr=lr, betas=(beta1, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=lr, betas=(beta1, 0.999))

# Function to visualize generated images
def visualize_generated_images(epoch, netG, nz=100, num_images=16):
    netG.eval()
    with torch.no_grad():
        noise = torch.randn(num_images, nz, 1, 1, device=device)
        fake_images = netG(noise).cpu()
    
    fake_images = fake_images * 0.5 + 0.5  # Denormalize to [0,1] range

    fig, axes = plt.subplots(4, 4, figsize=(8, 8))
    for i, ax in enumerate(axes.flatten()):
        img = np.transpose(fake_images[i].numpy(), (1, 2, 0))
        ax.imshow(img)
        ax.axis('off')

    plt.suptitle(f"Generated Images at Epoch {epoch}")
    plt.show()
    netG.train()

# Training Loop
num_epochs = 100
nz = 4  

for epoch in range(num_epochs):
    for i, real_images in enumerate(dataloader, 0):
        real_images = real_images.to(device)
        batch_size = real_images.size(0)

        # Train Discriminator
        netD.zero_grad()
        
        labels_real = torch.ones(batch_size, 1, device=device)
        output_real = netD(real_images)
        lossD_real = criterion(output_real, labels_real)
        
        noise = torch.randn(batch_size, nz, 32, 32, device=device)
        fake_images = netG(noise)
        
        labels_fake = torch.zeros(batch_size, 1, device=device)
        output_fake = netD(fake_images.detach())
        lossD_fake = criterion(output_fake, labels_fake)
        
        lossD = lossD_real + lossD_fake
        lossD.backward()
        optimizerD.step()

        # Train Generator
        netG.zero_grad()
        output_fake = netD(fake_images)
        lossG = criterion(output_fake, labels_real)  
        lossG.backward()
        optimizerG.step()

        # Print progress
        if i % 50 == 0:
            print(f"Epoch [{epoch}/{num_epochs}], Step [{i}/{len(dataloader)}], Loss D: {lossD.item()}, Loss G: {lossG.item()}")

    # Visualize generated images every 10 epochs
    if epoch % 10 == 0:
        visualize_generated_images(epoch, netG)

        # Save model checkpoints
        torch.save(netG.state_dict(), f"generator_epoch_{epoch}.pth")
        torch.save(netD.state_dict(), f"discriminator_epoch_{epoch}.pth")

# Save final models
torch.save(netG.state_dict(), "generator_final.pth")
torch.save(netD.state_dict(), "discriminator_final.pth")

# Generate and display final images
visualize_generated_images(num_epochs, netG)
